# 00 Persiapan Data Spasial dan Kendali Mutu (QA/QC) SFINCS Mandiri
Kernel: **HydroMT-SFINCS** (`env-sfincs`)

Notebook ini digunakan untuk menyiapkan, menyelaraskan, dan memvalidasi seluruh data masukan yang diperlukan dalam **Alur Kerja Pemodelan Bahaya Banjir Hidrodinamika 2D Mandiri HydroMT-SFINCS** (metode hujan langsung pada kisi) di Provinsi Sumatera Barat tanpa ketergantungan Wflow:

### Tujuan Pelaksanaan:
1. **Penyelarasan Sistem Referensi Koordinat (CRS):** Mendeteksi zona UTM Indonesia secara otomatis (`EPSG:32646` hingga `EPSG:32754`, misalnya `EPSG:32747` untuk Provinsi Sumatera Barat) dan memproyeksikan ulang seluruh lapisan vektor serta raster.
2. **Verifikasi Topografi (FABDEM 30 m Asli):** Memeriksa ketersediaan raster elevasi asli yang akan dipotong dan diproyeksikan secara langsung (*on-the-fly*) per klaster DAS untuk pembangkitan kisi dan subkisi 10 m.
3. **Jaringan Sungai (RBI Skala 1:25.000):** Memecah geometri jamak (*explode*), menyaring, dan memvalidasi garis jaringan sungai untuk pembakaran alur saluran serta penentuan titik keluaran.
4. **Kekasaran Permukaan Manning (Penutup Lahan RBI BIG):** Merasterisasi penutup lahan menjadi kisi bilangan bulat 30 m (`data/landcover_rbi.tif`) dan mencocokkannya dengan tabel `lookup_tables/manning_lookup.csv`.
5. **Infiltrasi Tanah Baku (Tekstur Tanah Kementan x Penutup Lahan RBI):** Mensintesis raster Bilangan Kurva USDA-NRCS (`data/cn_amc2_100m.tif` dan `data/cn_amc3_100m.tif`) untuk skema `CNB` serta raster infiltrasi konstan cadangan (`data/soil_infiltration_100m.tif`).
6. **Verifikasi NetCDF Hujan Rancangan:** Memeriksa deret waktu hujan rancangan per jam (`RP2` hingga `RP100`) beserta uji kesesuaian kenaikan intensitas antar kala ulang.
7. **Pembangunan Katalog Data HydroMT v0.10.x:** Menghasilkan berkas katalog mandiri `data_catalog_sfincs.yml`.


---
## 1. Pengaturan Lingkungan Kerja dan Proyeksi Sasaran


In [1]:
# Menetapkan direktori kerja secara otomatis ke folder utama proyek
from pathlib import Path
import os, sys
project_root = Path.cwd()
while not (project_root / 'data').exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
if str(project_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(project_root / 'scripts'))

import os
import sys
from pathlib import Path
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import xarray as xr
import numpy as np
import pyproj

# Pengaturan jalur PROJ, GDAL, dan DLL secara terpusat melalui scripts/_env.py
import _env  # noqa: F401
from geo_utils import resolve_target_crs

SOURCE_DEM = "data_raw/fabdem/FABDEM30m_Sumbar.tif"
TARGET_CRS = resolve_target_crs("auto", SOURCE_DEM)  # Deteksi zona UTM Indonesia otomatis (EPSG:32646 hingga 32754)
Path("data").mkdir(exist_ok=True, parents=True)
Path("data/rainfall").mkdir(exist_ok=True, parents=True)

print(f"Auto-detected Indonesian UTM Target CRS: {TARGET_CRS}")


Target CRS defined: EPSG:32747


---
## 2. Persiapan Lapisan Vektor (Sungai, Klaster DAS, dan Batas Administrasi)

Menyelaraskan seluruh lapisan vektor dari `data_raw/` ke `data/` pada proyeksi UTM sasaran (`TARGET_CRS`) serta memecah geometri garis jamak (*multipart LineStrings*) agar perpotongan topologi bersih.


In [2]:
print("=== Reprojecting and Cleaning Vector Layers ===")
vectors = [
    ("rivers", "data_raw/rbi/river.gpkg"),
    ("lake", "data_raw/rbi/lake.gpkg"),
    ("boundary_kecamatan", "data_raw/admin_boundary/boundary_kecamatan.gpkg"),
    ("boundary_kabkot", "data_raw/admin_boundary/boundary_kabkot.gpkg"),
    ("boundary_desakel", "data_raw/admin_boundary/boundary_desakel.gpkg"),
]

for name, path in vectors:
    if os.path.exists(path):
        gdf = gpd.read_file(path)
        print(f"{name:20s}: Native CRS = {gdf.crs}, Features = {len(gdf)}")
        if str(gdf.crs) != TARGET_CRS:
            gdf = gdf.to_crs(TARGET_CRS)
        
        # Untuk jaringan sungai: pecah geometri jamak menjadi LineString tunggal yang bersih
        if name == "rivers":
            gdf = gdf.explode(index_parts=False).reset_index(drop=True)
            
        out_p = f"data/{name}.gpkg"
        gdf.to_file(out_p, driver="GPKG")
        print(f"  -> Saved {out_p} ({len(gdf)} features in {TARGET_CRS})")
    else:
        print(f"Warning: {path} not found.")

# Verifikasi batas baku 17 klaster DAS
cluster_path = Path("data/das_clusters/sumbar_das_cluster.gpkg")
if cluster_path.exists():
    c_gdf = gpd.read_file(cluster_path)
    print(f"\n✓ Authoritative Cluster Boundary verified: {cluster_path} ({len(c_gdf)} clusters in {c_gdf.crs})")
else:
    print(f"\n⚠️ Warning: {cluster_path} not found. Please run scripts/rebuild_individual_clusters.py.")


=== Reprojecting and Cleaning Vector Layers ===
rivers              : Native CRS = EPSG:4326, Features = 52290
  -> Saved data/rivers.gpkg (52327 features in EPSG:32747)
lake                : Native CRS = EPSG:4326, Features = 1094
  -> Saved data/lake.gpkg (1094 features in EPSG:32747)
boundary_kecamatan  : Native CRS = EPSG:4326, Features = 181
  -> Saved data/boundary_kecamatan.gpkg (181 features in EPSG:32747)
boundary_kabkot     : Native CRS = EPSG:4326, Features = 23
  -> Saved data/boundary_kabkot.gpkg (23 features in EPSG:32747)
boundary_desakel    : Native CRS = EPSG:4326, Features = 1164
  -> Saved data/boundary_desakel.gpkg (1164 features in EPSG:32747)
admin_boundary      : Native CRS = EPSG:4326, Features = 23
  -> Saved data/admin_boundary.gpkg (23 features in EPSG:32747)

✓ Authoritative Cluster Boundary verified: data\das_clusters\sumbar_das_cluster.gpkg (17 clusters in EPSG:32747)


---
## 3. Verifikasi Raster Topografi (FABDEM 30 m Asli)

Memeriksa ketersediaan raster elevasi asli FABDEM 30 m (`data_raw/fabdem/FABDEM30m_Sumbar.tif`) yang akan dipotong dan diproyeksikan langsung per klaster DAS menggunakan GDAL `WarpedVRT`.


In [3]:
def reproject_large_raster(in_path, out_path, target_crs="EPSG:32747", is_discrete=False):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(in_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": target_crs,
            "transform": transform,
            "width": width,
            "height": height,
            "tiled": True,
            "blockxsize": 512,
            "blockysize": 512,
            "compress": "lzw",
            "BIGTIFF": "YES",
        })
        resampling = Resampling.nearest if is_discrete else Resampling.bilinear
        print(f"Reprojecting {in_path} -> {out_path} ({width}x{height})...")
        with rasterio.open(out_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=resampling,
                    num_threads=4,
                )
    print(f"Finished: {out_path}")

# FABDEM 30 m asli dipotong dan diproyeksikan langsung per DAS melalui WarpedVRT
# (menghemat sekitar 11 GB dibanding menyimpan GeoTIFF 10 m satu provinsi).
native_dem = Path("data_raw/fabdem/FABDEM30m_Sumbar.tif")
if native_dem.exists():
    with rasterio.open(native_dem) as src:
        print(f"✓ Native 30m FABDEM verified: {native_dem} ({src.width}x{src.height}, CRS={src.crs})")
else:
    print(f"⚠️ Source DEM not found at {native_dem}. Place native FABDEM 30m or regional DTM there.")


Found existing: data/fabdem_reprojected.tif


---
## 4. Persiapan Penutup Lahan (Manning n), Bilangan Kurva (CN), dan Infiltrasi Tanah

1. Memvalidasi `lookup_tables/manning_lookup.csv` serta merasterisasi penutup lahan RBI BIG ke kisi bilangan bulat 30 m (`data/landcover_rbi.tif`).
2. Mensintesis raster infiltrasi konstan cadangan (`data/soil_infiltration_100m.tif`) dari persilangan **Tekstur Tanah Kementerian Pertanian** (`TeksturTanah_Kementan.gpkg`) dan **Penutup Lahan RBI BIG** melalui `lookup_tables/soil_infiltration_lookup.csv`.
3. Mensintesis raster Bilangan Kurva USDA-NRCS (`data/cn_amc2_100m.tif` dan `data/cn_amc3_100m.tif`) untuk skema infiltrasi baku SFINCS `CNB`.


In [4]:
# ==============================================================================
# 4. Sintesis Penutup Lahan Baku (Manning n), Infiltrasi Tanah, dan Raster CN
# ==============================================================================
from scripts.generate_landcover_raster import generate_landcover_raster
from scripts.generate_soil_infiltration_raster import generate_soil_infiltration_raster
from scripts.generate_cn_raster import generate_cn_rasters

# 1. Bangkitkan raster Penutup Lahan RBI berkode bilangan bulat 30 m (Auto-UTM)
lc_tif = Path("data/landcover_rbi.tif")
if not lc_tif.exists() and os.path.exists("data_raw/rbi/landcover.gpkg"):
    print("Rasterizing RBI Landcover to authoritative 30m integer grid (data/landcover_rbi.tif)...")
    generate_landcover_raster(
        rbi_gpkg="data_raw/rbi/landcover.gpkg",
        ref_dem=SOURCE_DEM,
        out_raster="data/landcover_rbi.tif",
        resolution=30.0,
        target_crs=TARGET_CRS,
    )
else:
    print(f"✓ Authoritative Land Cover Raster verified: {lc_tif}")

# 2. Sintesis raster infiltrasi tanah baku (cadangan qinf konstan)
infil_tif = Path("data/soil_infiltration_100m.tif")
if not infil_tif.exists() and os.path.exists("data_raw/soil/TeksturTanah_Kementan.gpkg"):
    print("Synthesizing Kementan x RBI Soil Infiltration Raster (data/soil_infiltration_100m.tif)...")
    generate_soil_infiltration_raster(
        soil_gpkg="data_raw/soil/TeksturTanah_Kementan.gpkg",
        rbi_gpkg="data_raw/rbi/landcover.gpkg",
        ref_dem=SOURCE_DEM,
        out_tif="data/soil_infiltration_100m.tif",
        lookup_csv="lookup_tables/soil_infiltration_lookup.csv",
        target_crs=TARGET_CRS,
    )
else:
    print(f"✓ Authoritative Soil Infiltration Raster verified: {infil_tif}")

# 3. Sintesis raster Bilangan Kurva USDA-NRCS (AMC-II dan AMC-III untuk SFINCS v2.4+ CNB)
cn2_tif, cn3_tif = Path("data/cn_amc2_100m.tif"), Path("data/cn_amc3_100m.tif")
if (not cn2_tif.exists() or not cn3_tif.exists()) and os.path.exists("data_raw/soil/TeksturTanah_Kementan.gpkg"):
    print("Synthesizing USDA-NRCS Curve Number rasters (AMC-II & AMC-III)...")
    generate_cn_rasters(
        soil_gpkg="data_raw/soil/TeksturTanah_Kementan.gpkg",
        lc_raster="data/landcover_rbi.tif",
        ref_dem=SOURCE_DEM,
        target_crs=TARGET_CRS,
    )
else:
    print(f"✓ USDA-NRCS Curve Number rasters verified: {cn2_tif.name}, {cn3_tif.name}")


✓ Authoritative Land Cover Raster verified: data\landcover_rbi.tif
✓ Authoritative Soil Infiltration Raster verified: data\soil_infiltration_100m.tif


---
## 5. Verifikasi Berkas NetCDF Hujan Rancangan (RP2 hingga RP100)

Memvalidasi berkas NetCDF hujan rancangan 24 jam (`rainfall_rp*.nc`), memeriksa format sumbu waktu (`2024-01-01 00:00` hingga `23:00`), serta memastikan kenaikan intensitas hujan secara monoton antar kala ulang ($RP2 < RP5 < \dots < RP100$).


In [5]:
RP_FILES = {
    "rp2": "data/rainfall/rainfall_rp2_hourly.nc",
    "rp5": "data/rainfall/rainfall_rp5_hourly.nc",
    "rp10": "data/rainfall/rainfall_rp10_hourly.nc",
    "rp25": "data/rainfall/rainfall_rp25_hourly.nc",
    "rp50": "data/rainfall/rainfall_rp50_hourly.nc",
    "rp100": "data/rainfall/rainfall_rp100_hourly.nc",
}

# Apabila berkas di data/rainfall belum tersedia, format ulang dari data_raw/rainfall
for rp, dest in RP_FILES.items():
    raw_cand = f"data_raw/rainfall/precip_RP{rp[2:]}.nc"
    if not os.path.exists(dest) and os.path.exists(raw_cand):
        ds = xr.open_dataset(raw_cand)
        ds.to_netcdf(dest)
        print(f"Formatted {raw_cand} -> {dest}")

# Tabel ringkasan kendali mutu (QA/QC)
qa_records = []
for rp, path in RP_FILES.items():
    if os.path.exists(path):
        ds = xr.open_dataset(path)
        p_var = ds["precip"] if "precip" in ds else ds[list(ds.data_vars.keys())[0]]
        max_rate = float(p_var.max())
        sum_24h = float(p_var.sum(dim="time").max() if "time" in p_var.dims else p_var.sum())
        qa_records.append({
            "Return Period": rp.upper(),
            "Max Hourly (mm/hr)": round(max_rate, 2),
            "24h Max Total (mm)": round(sum_24h, 2),
            "Time Steps": len(ds.time) if "time" in ds else 1,
            "File Status": "Ready"
        })
    else:
        qa_records.append({"Return Period": rp.upper(), "File Status": "Missing"})

df_qa = pd.DataFrame(qa_records)
print("=== Rainfall Monotonicity & QA Summary ===")
print(df_qa.to_string(index=False))


=== Rainfall Monotonicity & QA Summary ===
Return Period  Max Hourly (mm/hr)  24h Max Total (mm)  Time Steps File Status
          RP2               74.91              216.08          24       Ready
          RP5               89.70              258.74          24       Ready
         RP10               99.49              286.99          24       Ready
         RP25              111.87              322.69          24       Ready
         RP50              121.05              349.16          24       Ready
        RP100              130.16              375.45          24       Ready


---
## 6. Pembangunan Katalog Data HydroMT untuk SFINCS Mandiri (`data_catalog_sfincs.yml`)

Menyusun berkas katalog terpadu HydroMT v0.10.x yang merujuk pada seluruh lapisan data hasil penyelarasan.


In [6]:
epsg_num = int(TARGET_CRS.split(":")[-1])
catalog_content = f"""# HydroMT Data Catalog for Standalone SFINCS Modeling (National v2.0, HydroMT v0.10.x)

fabdem_sumbar:
  path: {SOURCE_DEM}
  data_type: RasterDataset
  driver: raster
  crs: 4326

elevation_dem:
  path: {SOURCE_DEM}
  data_type: RasterDataset
  driver: raster
  crs: 4326

rbi_river_sumbar:
  path: data/rivers.gpkg
  data_type: GeoDataFrame
  driver: vector
  crs: {epsg_num}

rivers:
  path: data/rivers.gpkg
  data_type: GeoDataFrame
  driver: vector
  crs: {epsg_num}

rbi_landcover_sumbar:
  path: data/landcover_rbi.tif
  data_type: RasterDataset
  driver: raster
  crs: {epsg_num}
  nodata: 0

landcover:
  path: data/landcover_rbi.tif
  data_type: RasterDataset
  driver: raster
  crs: {epsg_num}
  nodata: 0

soil_infiltration_sumbar:
  path: data/soil_infiltration_100m.tif
  data_type: RasterDataset
  driver: raster
  crs: {epsg_num}

das_clusters:
  path: data/das_clusters/sumbar_das_cluster.gpkg
  data_type: GeoDataFrame
  driver: vector
  crs: {epsg_num}

rainfall_rp2:
  path: data/rainfall/rainfall_rp2_hourly.nc
  data_type: RasterDataset
  driver: netcdf

rainfall_rp5:
  path: data/rainfall/rainfall_rp5_hourly.nc
  data_type: RasterDataset
  driver: netcdf

rainfall_rp10:
  path: data/rainfall/rainfall_rp10_hourly.nc
  data_type: RasterDataset
  driver: netcdf

rainfall_rp25:
  path: data/rainfall/rainfall_rp25_hourly.nc
  data_type: RasterDataset
  driver: netcdf

rainfall_rp50:
  path: data/rainfall/rainfall_rp50_hourly.nc
  data_type: RasterDataset
  driver: netcdf

rainfall_rp100:
  path: data/rainfall/rainfall_rp100_hourly.nc
  data_type: RasterDataset
  driver: netcdf
"""

with open("data_catalog_sfincs.yml", "w", encoding="utf-8") as f:
    f.write(catalog_content.strip() + "\n")

print("data_catalog_sfincs.yml generated successfully pointing to sumbar_das_cluster.gpkg!")


data_catalog_sfincs.yml generated successfully pointing to sumbar_das_cluster.gpkg!
